# 37 - Benchmark: CK+ Dataset

**Dataset:** CK+ — 636/654 gambar, 7/8 emosi, 118 subjek
**Evaluasi:** 10-Fold CV (subject-wise, 118 subjek terlalu banyak untuk LOSO)
**Model:** Sama dengan eksperimen utama (CNN, FCNN, Late Fusion, Intermediate, + TL)
**Skenario:** B1 (Baseline), B2 (Class Weights), B3 (Augmented)
**Konfigurasi kelas:**
- 7-class (drop contempt, 636 sampel)
- 4-class (contempt → negative, 654 sampel)

In [ ]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
)
from training.utils import train_model, full_evaluation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

BATCH_SIZE = 16
EPOCHS = 50
PATIENCE = 15
K_FOLDS = 10
SEED = 42
OUTPUT_DIR = PROJECT_ROOT / "models" / "benchmark" / "ckplus"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = [
    ("CNN", EmotionCNN, "cnn", 0.0001),
    ("FCNN", EmotionFCNN, "fcnn", 0.0001),
    ("Intermediate", IntermediateFusion, "fusion", 0.0001),
    ("CNN_TL", EmotionCNNTransfer, "cnn", 0.00005),
    ("Intermediate_TL", IntermediateFusionTransfer, "fusion", 0.00005),
]

print("Setup complete.")

In [ ]:
# ── Helper Functions (same training helpers as notebook 36) ──

def make_loader(images, landmarks, labels, model_type, batch_size=16, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(landmarks)
    y_t = torch.from_numpy(labels).long()
    if model_type == "cnn": ds = TensorDataset(img_t, y_t)
    elif model_type == "fcnn": ds = TensorDataset(lm_t, y_t)
    else: ds = TensorDataset(img_t, lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def augment_train(images, landmarks, labels, target_min=20):
    import cv2
    counts = Counter(labels.tolist())
    aug_img, aug_lm, aug_y = list(images), list(landmarks), list(labels)
    for cls_idx, cls_count in counts.items():
        if cls_count >= target_min: continue
        needed = target_min - cls_count
        cls_indices = np.where(labels == cls_idx)[0]
        added = 0
        while added < needed:
            for idx in cls_indices:
                if added >= needed: break
                img = (images[idx] * 255).astype(np.uint8)
                flipped = cv2.flip(img, 1)
                aug_img.append(flipped.astype(np.float32) / 255.0)
                lm = landmarks[idx].reshape(68, 2).copy(); lm[:, 0] = 1.0 - lm[:, 0]
                aug_lm.append(lm.flatten()); aug_y.append(cls_idx); added += 1
    return np.array(aug_img), np.array(aug_lm), np.array(aug_y)


def get_class_weights(labels, device):
    counts = Counter(labels.tolist()); n_classes = len(counts); total = len(labels)
    weights = torch.zeros(n_classes)
    for c, cnt in counts.items(): weights[c] = total / (n_classes * cnt)
    return weights.to(device)


def train_fold_standard(ModelClass, model_type, lr, criterion,
                        train_img, train_lm, train_y, test_img, test_lm, test_y,
                        emotions, fold_dir):
    n_val = max(1, int(len(train_y) * 0.15))
    perm = np.random.RandomState(42).permutation(len(train_y))
    val_i, tr_i = perm[:n_val], perm[n_val:]
    tr_loader = make_loader(train_img[tr_i], train_lm[tr_i], train_y[tr_i], model_type, BATCH_SIZE)
    val_loader = make_loader(train_img[val_i], train_lm[val_i], train_y[val_i], model_type, BATCH_SIZE, False)
    test_loader = make_loader(test_img, test_lm, test_y, model_type, BATCH_SIZE, False)
    model = ModelClass(num_classes=len(emotions)).to(device)
    save_path = str(fold_dir / "model.pth")
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(model, tr_loader, val_loader, criterion, optimizer, scheduler,
                device, model_type, EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, criterion, device, model_type, emotions)
    os.remove(save_path)
    return {"accuracy": float(r["test_accuracy"]), "macro_f1": float(r["test_macro_f1"]),
            "weighted_f1": float(r["test_weighted_f1"])}


def late_fusion_fold(train_img, train_lm, train_y, test_img, test_lm, test_y, num_classes, fold_dir):
    n_val = max(1, int(len(train_y) * 0.15))
    perm = np.random.RandomState(42).permutation(len(train_y))
    val_i, tr_i = perm[:n_val], perm[n_val:]
    cnn = EmotionCNN(num_classes=num_classes).to(device)
    cnn_tr = make_loader(train_img[tr_i], train_lm[tr_i], train_y[tr_i], "cnn", BATCH_SIZE)
    cnn_val = make_loader(train_img[val_i], train_lm[val_i], train_y[val_i], "cnn", BATCH_SIZE, False)
    opt = torch.optim.Adam(cnn.parameters(), lr=0.0001)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(cnn, cnn_tr, cnn_val, nn.CrossEntropyLoss(), opt, sch, device, "cnn", EPOCHS, PATIENCE, str(fold_dir/"cnn.pth"))
    fcnn = EmotionFCNN(num_classes=num_classes).to(device)
    fcnn_tr = make_loader(train_img[tr_i], train_lm[tr_i], train_y[tr_i], "fcnn", BATCH_SIZE)
    fcnn_val = make_loader(train_img[val_i], train_lm[val_i], train_y[val_i], "fcnn", BATCH_SIZE, False)
    opt2 = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
    sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(fcnn, fcnn_tr, fcnn_val, nn.CrossEntropyLoss(), opt2, sch2, device, "fcnn", EPOCHS, PATIENCE, str(fold_dir/"fcnn.pth"))
    cnn.load_state_dict(torch.load(fold_dir/"cnn.pth", map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(fold_dir/"fcnn.pth", map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()
    test_img_t = torch.from_numpy(test_img).permute(0,3,1,2).to(device)
    test_lm_t = torch.from_numpy(test_lm).to(device)
    with torch.no_grad():
        cnn_probs = torch.softmax(cnn(test_img_t), dim=1).cpu().numpy()
        fcnn_probs = torch.softmax(fcnn(test_lm_t), dim=1).cpu().numpy()
    best_f1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        preds = (w*cnn_probs + (1-w)*fcnn_probs).argmax(axis=1)
        f1 = f1_score(test_y, preds, average="macro", zero_division=0)
        if f1 > best_f1: best_f1=f1; best_w=w; best_preds=preds
    acc = accuracy_score(test_y, best_preds)
    wf1 = f1_score(test_y, best_preds, average="weighted", zero_division=0)
    for f in ["cnn.pth","fcnn.pth"]: (fold_dir/f).unlink(missing_ok=True)
    return {"accuracy": acc, "macro_f1": best_f1, "weighted_f1": wf1, "best_cnn_weight": best_w}


def run_benchmark_kfold(dataset_name, data_dir, num_classes, emotions):
    """Run 10-fold subject-wise CV for all models on CK+."""
    print(f"\n{'='*70}")
    print(f"  BENCHMARK: {dataset_name} ({num_classes}-class, {K_FOLDS}-Fold CV)")
    print(f"{'='*70}")

    images = np.load(data_dir / "X_images.npy")
    landmarks = np.load(data_dir / "X_landmarks.npy")
    labels = np.load(data_dir / "y_labels.npy")
    subjects = np.load(data_dir / "subjects.npy", allow_pickle=True)

    unique_subjects = sorted(set(subjects))
    subject_indices = {s: np.where(subjects == s)[0] for s in unique_subjects}

    # Create k-fold split (subject-wise)
    rng = np.random.RandomState(SEED)
    subj_arr = np.array(unique_subjects)
    rng.shuffle(subj_arr)
    folds = np.array_split(subj_arr, K_FOLDS)

    print(f"  Samples: {len(labels)}, Subjects: {len(unique_subjects)}, Folds: {K_FOLDS}")
    print(f"  Distribution: {dict(sorted(Counter(labels.tolist()).items()))}")
    for fi, fold_subjs in enumerate(folds):
        n = sum(len(subject_indices[s]) for s in fold_subjs)
        print(f"    Fold {fi+1}: {len(fold_subjs)} subjects, {n} samples")

    all_results = {}
    models_to_run = MODELS + [("Late_Fusion", None, "late", 0.0001)]

    for model_name, ModelClass, model_type, lr in models_to_run:
        for scenario in ["B1", "B2", "B3"]:
            key = f"{model_name}_{scenario}"
            print(f"\n  >> {key} ({K_FOLDS} folds)")

            fold_results = []
            model_dir = OUTPUT_DIR / f"{dataset_name}_{num_classes}c" / key
            os.makedirs(model_dir, exist_ok=True)

            for fi in range(K_FOLDS):
                test_subjects = folds[fi]
                train_subjects = np.concatenate([folds[j] for j in range(K_FOLDS) if j != fi])
                test_idx = np.concatenate([subject_indices[s] for s in test_subjects])
                train_idx = np.concatenate([subject_indices[s] for s in train_subjects])

                train_img, train_lm, train_y = images[train_idx], landmarks[train_idx], labels[train_idx]
                test_img, test_lm, test_y = images[test_idx], landmarks[test_idx], labels[test_idx]

                if scenario == "B2":
                    weights = get_class_weights(train_y, device)
                    criterion = nn.CrossEntropyLoss(weight=weights)
                elif scenario == "B3":
                    train_img, train_lm, train_y = augment_train(train_img, train_lm, train_y)
                    weights = get_class_weights(train_y, device)
                    criterion = nn.CrossEntropyLoss(weight=weights)
                else:
                    criterion = nn.CrossEntropyLoss()

                fold_dir = model_dir / f"fold_{fi}"
                os.makedirs(fold_dir, exist_ok=True)

                if model_type == "late":
                    r = late_fusion_fold(train_img, train_lm, train_y, test_img, test_lm, test_y, num_classes, fold_dir)
                else:
                    r = train_fold_standard(ModelClass, model_type, lr, criterion,
                                            train_img, train_lm, train_y, test_img, test_lm, test_y,
                                            emotions, fold_dir)
                fold_results.append(r)
                try: fold_dir.rmdir()
                except: pass

            f1s = [r["macro_f1"] for r in fold_results]
            accs = [r["accuracy"] for r in fold_results]
            print(f"     F1: {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}  "
                  f"Acc: {np.mean(accs):.4f} +/- {np.std(accs):.4f}")

            all_results[key] = {
                "model": model_name, "scenario": scenario,
                "macro_f1_mean": float(np.mean(f1s)), "macro_f1_std": float(np.std(f1s)),
                "accuracy_mean": float(np.mean(accs)), "accuracy_std": float(np.std(accs)),
                "k_folds": K_FOLDS, "per_fold": fold_results,
            }

    save_path = OUTPUT_DIR / f"{dataset_name}_{num_classes}c_results.json"
    with open(save_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\n  Saved: {save_path}")
    return all_results

print("Helper functions ready.")

## Run CK+ Benchmark

In [ ]:
BENCHMARK_DIR = PROJECT_ROOT / "data" / "benchmark"
EMOTIONS_7 = ["neutral", "happy", "sad", "angry", "fearful", "disgusted", "surprised"]
EMOTIONS_4 = ["neutral", "happy", "sad", "negative"]

# 7-class CK+ (tanpa contempt)
res_7c = run_benchmark_kfold("ckplus", BENCHMARK_DIR / "ckplus_7class", 7, EMOTIONS_7)

# 4-class CK+ (contempt → negative)
res_4c = run_benchmark_kfold("ckplus", BENCHMARK_DIR / "ckplus_4class_contempt", 4, EMOTIONS_4)

## Ringkasan CK+

In [ ]:
# Summary table
for nc_label, res in [("7-class", res_7c), ("4-class", res_4c)]:
    print(f"\n{'='*70}")
    print(f"  CK+ {nc_label} - 10-Fold CV Results (sorted by Macro F1)")
    print(f"{'='*70}")
    print(f"  {'Model + Scenario':<30} {'Macro F1':>18} {'Accuracy':>18}")
    print(f"  {'-'*68}")
    for key in sorted(res.keys(), key=lambda k: -res[k]["macro_f1_mean"]):
        r = res[key]
        print(f"  {key:<30} {r['macro_f1_mean']:.4f} +/- {r['macro_f1_std']:.4f}"
              f"  {r['accuracy_mean']:.4f} +/- {r['accuracy_std']:.4f}")

# Compare with own dataset
print(f"\n{'='*70}")
print(f"  PERBANDINGAN: CK+ vs Dataset Sendiri (Front-Only)")
print(f"{'='*70}")
print(f"  {'Model':<25} {'CK+ 7c':>15} {'Own 7c':>15} {'CK+ 4c':>15} {'Own 4c':>15}")
print(f"  {'-'*68}")

own_results = {
    "CNN_B1": {"7c": 0.137, "4c": 0.238},
    "FCNN_B1": {"7c": 0.158, "4c": 0.307},
    "Intermediate_B1": {"7c": 0.137, "4c": 0.243},
    "CNN_TL_B1": {"7c": 0.154, "4c": 0.274},
    "Intermediate_TL_B1": {"7c": 0.173, "4c": 0.412},
}

for model_key, own in own_results.items():
    c7 = res_7c.get(model_key, {}).get("macro_f1_mean", 0)
    c4 = res_4c.get(model_key, {}).get("macro_f1_mean", 0)
    print(f"  {model_key:<25} {c7:>15.4f} {own['7c']:>15.4f} {c4:>15.4f} {own['4c']:>15.4f}")